In [ ]:
# 1. Install necessary libraries (Run this once)
!pip install -q -U transformers accelerate bitsandbytes

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm.auto import tqdm

# ==========================================
# 1. SETUP MODEL
# ==========================================
# PASTE YOUR TOKEN BELOW
HF_TOKEN = "enter token here"

# We use Llama 3.1 8B Instruct (Good balance of speed and smarts)
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

# Load in 4-bit to fit on Kaggle GPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=None, # You can use BitsAndBytesConfig if memory is tight
)

# Create the pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,       # Enough for a list of species
    temperature=0.1,          # Low temperature = More factual, less creative
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

# ==========================================
# 2. DEFINE THE PROMPT
# ==========================================
def get_species_from_llm(text):
    # System prompt tells the model who it is
    system_prompt = "You are a biological expert. Extract scientific species names from the text."

    # User prompt gives the specific task
    user_prompt = f"""
    Analyze the following text and extract all scientific species names (Latin names).

    Rules:
    1. Return ONLY a comma-separated list.
    2. Do not write sentences.
    3. If no species are found, return "None".
    4. Ignore common names (like "red deer"), only keep Latin names (like "Cervus elaphus").

    Text: "{text}"

    Scientific Names:
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Run Model
    outputs = pipe(messages)
    result = outputs[0]['generated_text']
    return result.strip()

# ==========================================
# 3. RUN ON YOUR DATA
# ==========================================
# Load your small dataset
df = pd.read_csv("Data_cleaned.csv")

# Combine Title + Description for context
df['text'] = df['Title'].fillna('') + " " + df['Description'].fillna('')

# Test on the first 10 rows to save time
subset = df.head(10).copy()

print("Running Extraction...")
tqdm.pandas(desc="LLM Extraction")
subset['LLM_Extracted'] = subset['text'].progress_apply(get_species_from_llm)

# ==========================================
# 4. VIEW RESULTS
# ==========================================
print("\n=== EXTRACTION RESULTS ===")
for i, row in subset.iterrows():
    print(f"Row {i}:")
    print(f"Ground Truth: {row.get('Species', 'Unknown')}")
    print(f"LLM Prediction: {row['LLM_Extracted']}")
    print("-" * 30)

# Save to CSV if you want
subset.to_csv("llama3_extraction_results.csv", index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00
Loading meta-llama/Meta-Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Running Extraction...


LLM Extraction:   0%|          | 0/10 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


=== EXTRACTION RESULTS ===
Row 0:
Ground Truth: Ostrea edulis

LLM Prediction: Ostrea edulis
------------------------------
Row 1:
Ground Truth: Epinephelus marginatus, Sciaena umbra, Diplodus cervinus
LLM Prediction: Epinephelus marginatus, Sciaena umbra, Diplodus cervinus
------------------------------
Row 2:
Ground Truth: Lyallia kerguelensis
LLM Prediction: Lyallia kerguelensis
------------------------------
Row 3:
Ground Truth: Trisopterus minutus

LLM Prediction: Trisopterus minutus
------------------------------
Row 4:
Ground Truth: nan
LLM Prediction: None
------------------------------
Row 5:
Ground Truth: nan
LLM Prediction: None
------------------------------
Row 6:
Ground Truth: nan
LLM Prediction: None
------------------------------
Row 7:
Ground Truth: nan
LLM Prediction: None
------------------------------
Row 8:
Ground Truth: nan
LLM Prediction: None
------------------------------
Row 9:
Ground Truth: nan
LLM Prediction: None
------------------------------


In [2]:
# ==========================================
# 2. DEFINE PROMPT & EXTRACTOR
# ==========================================
def get_species_from_llm(text):
    system_prompt = "You are a biological expert. Extract scientific species names from the text."
    user_prompt = f"""
    Analyze the text and extract all scientific species names (Latin names).

    Rules:
    1. Return ONLY a comma-separated list.
    2. Do not write sentences.
    3. If no species are found, return "None".
    4. Ignore common names, only keep Latin names.

    Text: "{text}"

    Scientific Names:
    """

    # Check if model supports system roles (Llama 3 does, Mistral v0.2 uses [INST])
    if "Llama-3" in MODEL_ID:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        outputs = pipe(messages)
        result = outputs[0]['generated_text']
    else:
        # Generic/Mistral formatting
        prompt = f"[INST] {system_prompt}\n{user_prompt} [/INST]"
        outputs = pipe(prompt)
        result = outputs[0]['generated_text']

    return result.strip()

# ==========================================
# 3. METRIC CALCULATION LOGIC
# ==========================================
def calculate_metrics(row):
    # 1. Clean Ground Truth
    gt_raw = str(row.get('Species', ''))
    if gt_raw.lower() == 'nan' or gt_raw.strip() == '':
        y_true = set()
    else:
        y_true = set([s.strip().lower() for s in gt_raw.split(',') if s.strip()])

    # 2. Clean LLM Prediction
    pred_raw = str(row.get('LLM_Extracted', ''))
    # Remove common LLM "None" outputs
    if pred_raw.lower() in ['none', 'none.', 'no species found']:
        y_pred = set()
    else:
        # Split by comma, strip, lowercase, remove trailing periods
        y_pred = set([s.strip().lower().rstrip('.') for s in pred_raw.split(',') if s.strip()])

    # 3. Calculate Matches (Intersection)
    tp = len(y_true.intersection(y_pred))
    fp = len(y_pred - y_true)
    fn = len(y_true - y_pred)

    # 4. Scores
    precision = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if len(y_pred) == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # Accuracy (Jaccard Index for sets)
    union = len(y_true.union(y_pred))
    accuracy = tp / union if union > 0 else 1.0

    return pd.Series([precision, recall, f1, accuracy, tp, fp, fn])

# ==========================================
# 4. RUN ON FULL DATASET
# ==========================================
# Load Data
df = pd.read_csv("Data_cleaned.csv")
df['text'] = df['Title'].fillna('') + " " + df['Description'].fillna('')

print(f"Processing {len(df)} rows on GPU...")

# Run Extraction (FULL DATASET)
tqdm.pandas(desc="LLM Extraction")
df['LLM_Extracted'] = df['text'].progress_apply(get_species_from_llm)

# Calculate Metrics
print("Calculating Metrics...")
metrics_df = df.apply(calculate_metrics, axis=1)
metrics_df.columns = ['Precision', 'Recall', 'F1', 'Accuracy', 'TP', 'FP', 'FN']
df_final = pd.concat([df, metrics_df], axis=1)

# ==========================================
# 5. FINAL REPORT
# ==========================================
print("\n" + "="*40)
print(f"RESULTS FOR MODEL: {MODEL_ID}")
print("="*40)

# Macro Averages
print(f"Macro Precision: {df_final['Precision'].mean():.4f}")
print(f"Macro Recall:    {df_final['Recall'].mean():.4f}")
print(f"Macro F1 Score:  {df_final['F1'].mean():.4f}")

# Global (Micro) Averages
total_tp = df_final['TP'].sum()
total_fp = df_final['FP'].sum()
total_fn = df_final['FN'].sum()

global_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
global_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
global_f1 = 2 * (global_p * global_r) / (global_p + global_r) if (global_p + global_r) > 0 else 0

print("-" * 30)
print(f"Total Correct (TP): {int(total_tp)}")
print(f"Total Wrong (FP):   {int(total_fp)}")
print(f"Total Missed (FN):  {int(total_fn)}")
print("-" * 30)
print(f"Global Precision:   {global_p:.4f}")
print(f"Global Recall:      {global_r:.4f}")
print(f"Global F1 Score:    {global_f1:.4f}")
print("="*40)

# Save
df_final.to_csv("llm_full_results.csv", index=False)
print("Saved full results to 'llm_full_results.csv'")

Processing 44 rows on GPU...


LLM Extraction:   0%|          | 0/44 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati

Calculating Metrics...

RESULTS FOR MODEL: meta-llama/Meta-Llama-3.1-8B-Instruct
Macro Precision: 0.8743
Macro Recall:    0.9390
Macro F1 Score:  0.8735
------------------------------
Total Correct (TP): 58
Total Wrong (FP):   21
Total Missed (FN):  5
------------------------------
Global Precision:   0.7342
Global Recall:      0.9206
Global F1 Score:    0.8169
Saved full results to 'llm_full_results.csv'
